In [ ]:
!nvidia-smi
!python --version

Mon Sep  7 16:55:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
ZIP = '/content/drive/MyDrive/tensors_messy_packed.zip'
!ls -lh "$ZIP"
!mkdir -p /content/data
!unzip -q "$ZIP" -d /content/data
!ls /content/data

-rw------- 1 root root 580M Sep  7 16:39 /content/drive/MyDrive/tensors_messy_packed.zip
tensors_messy_packed


In [ ]:
!apt-get install -qq python3.11 python3.11-venv python3.11-dev > /dev/null 2>&1
!python3.11 -m venv /content/akv
!/content/akv/bin/pip install -q --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 15.4 MB/s eta 0:00:00


In [ ]:
!/content/akv/bin/pip install -q akida-models==1.14.2

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
!MPLBACKEND=Agg /content/akv/bin/python -c "import tensorflow as tf; print(tf.__version__); print(tf.config.list_physical_devices('GPU'))"

2026-09-07 17:00:15.256232: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788800415.285742    3462 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788800415.294177    3462 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1788800415.313215    3462 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788800415.313260    3462 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788800415.313265    3462 computation_placer.cc:177] computation placer alr

In [ ]:
from google.colab import files
uploaded = files.upload()
!ls *.py

Saving config.py to config.py
Saving dataset.py to dataset.py
config.py  dataset.py


In [ ]:
from google.colab import files
uploaded = files.upload()
!ls *.py

Saving targets.py to targets.py
Saving train.py to train.py
config.py  dataset.py  targets.py  train.py


In [ ]:
import re

src = open('config.py').read()
src = re.sub(r"ROOT = Path\(.*?\)", "ROOT = Path('/content')", src)
src = re.sub(r"TENSOR_DIR = .*", "TENSOR_DIR = Path('/content/data/tensors_messy_packed')", src)
src = re.sub(r"SPLITS_FILE = .*", "SPLITS_FILE = Path('/content/data/tensors_messy_packed/splits.json')", src)
src = re.sub(r"RUNS_DIR = .*", "RUNS_DIR = Path('/content/runs')", src)
open('config.py','w').write(src)

src = open('dataset.py').read()
src = src.replace(
    'if not npy.exists() or not meta_path.exists():',
    'npz = tensor_dir / f"{clip}_tensors.npz"\n    if not meta_path.exists() or (not npy.exists() and not npz.exists()):'
)
src = src.replace(
    'tensors = np.load(npy, mmap_mode="r")',
    'tensors = np.load(npy, mmap_mode="r") if npy.exists() else np.load(npz)["a"]'
)
open('dataset.py','w').write(src)

print(open('config.py').read()[:600])

"""
Every constant that has to agree across the pipeline.

Anchors in particular must be identical in target creation and in
inference decoding. If they drift apart the network trains fine and
then predicts boxes of the wrong size, which looks like a broken model
rather than a broken constant. Keeping them in one file that both sides
import removes that failure mode.
"""

from pathlib import Path


# ============================================================
# PATHS
# ============================================================

ROOT = Path('/content')

TENSOR_DIR = Path('/content/data/tenso


In [ ]:
!MPLBACKEND=Agg /content/akv/bin/python dataset.py

tensors : /content/data/tensors_messy_packed
policy  : keep

TRAIN  (90 clips in split)
  clips loaded : 90
  samples      : 28045
  quiet frames : 1184 (policy: keep)
  boxes        : 28345
  box size     : median 10.4 px (0.32 cells), p5 6.7, p95 17.7
  under 8 px   : 16.4%

VALIDATION  (12 clips in split)
  clips loaded : 12
  samples      : 3728
  quiet frames : 82 (policy: keep)
  boxes        : 3728
  box size     : median 10.2 px (0.32 cells), p5 6.5, p95 18.2
  under 8 px   : 25.0%

TEST  (12 clips in split)
  clips loaded : 12
  samples      : 3723
  quiet frames : 19 (policy: keep)
  boxes        : 4243
  box size     : median 10.6 px (0.33 cells), p5 8.1, p95 20.5
  under 8 px   : 4.6%

One batch:
  images shape : (4, 224, 224, 1)  float32
  value range  : -6.64 to 10.93
  zero pixels  : 87.7%
  boxes/frame  : [1, 1, 1, 1]
  out of bounds: 0
  inside padding: 0


In [ ]:
import subprocess, threading, time, os

os.makedirs('/content/drive/MyDrive/drone_runs', exist_ok=True)

# Copy results to Drive every 10 minutes, so a disconnect at hour 6
# does not cost you the run. train.py rewrites history.json after every
# epoch and best.weights.h5 whenever validation improves.
def backup():
    while True:
        time.sleep(600)
        subprocess.run('cp -r /content/runs/* /content/drive/MyDrive/drone_runs/ 2>/dev/null',
                       shell=True)

threading.Thread(target=backup, daemon=True).start()

env = dict(os.environ, MPLBACKEND='Agg')

subprocess.run([
    '/content/akv/bin/python', 'train.py',
    '--epochs', '30',
    '--lr', '1e-3',
    '--batch_size', '64',
    '--name', 'full_20ms',
], env=env)

subprocess.run('cp -r /content/runs/* /content/drive/MyDrive/drone_runs/', shell=True)
print('done, copied to Drive')

done, copied to Drive


In [ ]:
import json
h = json.load(open('/content/runs/full_20ms/history.json'))
for e in h['history']:
    print(f"{e['epoch']:3d}  train {e['train_loss']:7.3f}  val {e['val_loss']:7.3f}  "
          f"t.rec {e['train_recall']:.3f}  v.rec {e['val_recall']:.3f}  "
          f"maxobj {e['val_max_objectness']:.3f}")

  1  train   3.418  val   6.023  t.rec 0.579  v.rec 0.119  maxobj 0.898
  2  train   1.347  val   2.339  t.rec 0.869  v.rec 0.618  maxobj 0.992
  3  train   1.157  val   2.286  t.rec 0.893  v.rec 0.629  maxobj 0.993
  4  train   1.023  val   2.183  t.rec 0.908  v.rec 0.631  maxobj 0.998
  5  train   0.924  val   2.189  t.rec 0.919  v.rec 0.612  maxobj 0.999
  6  train   0.836  val   1.941  t.rec 0.927  v.rec 0.663  maxobj 0.999
  7  train   0.752  val   2.121  t.rec 0.936  v.rec 0.647  maxobj 1.000
  8  train   0.691  val   2.180  t.rec 0.944  v.rec 0.629  maxobj 1.000
  9  train   0.631  val   2.127  t.rec 0.947  v.rec 0.626  maxobj 1.000
 10  train   0.563  val   2.549  t.rec 0.954  v.rec 0.544  maxobj 1.000
 11  train   0.520  val   2.484  t.rec 0.958  v.rec 0.543  maxobj 1.000
 12  train   0.474  val   2.559  t.rec 0.961  v.rec 0.545  maxobj 1.000
 13  train   0.431  val   2.532  t.rec 0.966  v.rec 0.543  maxobj 1.000
 14  train   0.400  val   2.773  t.rec 0.967  v.rec 0.491  maxob

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving evaluate.py to evaluate.py


In [ ]:
!MPLBACKEND=Agg /content/akv/bin/python evaluate.py --run full_20ms --split validation

2026-09-07 20:10:58.715399: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788811858.738761   51328 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788811858.750112   51328 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1788811858.777223   51328 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788811858.777266   51328 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788811858.777272   51328 computation_placer.cc:177] computation placer alr

In [ ]:
!ls -lh /content/drive/MyDrive/*.zip

-rw------- 1 root root 566M Sep  7 20:27 /content/drive/MyDrive/tensors_messy_10ms_packed.zip
-rw------- 1 root root 1.1G Sep  7 20:28 /content/drive/MyDrive/tensors_messy_40ms_packed.zip
-rw------- 1 root root 580M Sep  7 16:39 /content/drive/MyDrive/tensors_messy_packed.zip


In [ ]:
for name in ['tensors_messy_10ms_packed', 'tensors_messy_40ms_packed']:
    !unzip -q "/content/drive/MyDrive/{name}.zip" -d /content/data
!ls /content/data

tensors_messy_10ms_packed  tensors_messy_40ms_packed  tensors_messy_packed


In [ ]:
!MPLBACKEND=Agg /content/akv/bin/python train.py --epochs 10 --lr 1e-3 --batch_size 64 --tensors /content/data/tensors_messy_10ms_packed --name full_10ms

2026-09-07 20:46:16.659047: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788813976.681453   60874 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788813976.688163   60874 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1788813976.704150   60874 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788813976.704196   60874 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1788813976.704200   60874 computation_placer.cc:177] computation placer alr

In [ ]:
!MPLBACKEND=Agg /content/akv/bin/python train.py --epochs 10 --lr 1e-3 --batch_size 64 --tensors /content/data/tensors_messy_40ms_packed --name full_40ms